# FX-Analyse

Interaktive Fassung der täglichen Auswertung. Die Logik liegt in `src/fx_analysis/`,
damit Notebook und GitHub Action exakt denselben Code ausführen.

**Fragestellung:** Wie haben sich die wichtigsten Währungen aus Sicht eines Euro-Anlegers
über verschiedene Zeiträume entwickelt, und in welchem Trend befinden sie sich aktuell?

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from fx_analysis import analysis, charts, store
from fx_analysis.config import CHART_PNG

## Datenbasis

EZB-Referenzkurse (täglicher Voll-Abruf ab 1999), gespeichert als `data/fx_timeseries.csv`.

In [ ]:
history = store.load()
matrix = store.wide(history)
print(f"{len(history):,} Zeilen, {matrix.index.min():%d.%m.%Y} – {matrix.index.max():%d.%m.%Y}")
matrix.tail()

## Analyse

Kurse werden in Euro umgerechnet (Wert von 1 Einheit Fremdwährung in EUR). Positive Werte bedeuten: die Währung hat gegenüber dem Euro aufgewertet.

In [ ]:
result = analysis.run(matrix)
result.returns.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", axis=0)

In [ ]:
result.trend

In [ ]:
print(analysis.interpretation(result))

## Grafik

In [ ]:
from IPython.display import Image

charts.render(result, CHART_PNG, "Notebook-Lauf")
Image(CHART_PNG)

## Währungsquote – Über-/Untergewichtung mit Backtest

Jede Währung des Korbs (Euro als Numeraire plus die zehn Währungen oben) wird gegen ihr eigenes gleitendes
Mittel gemessen – der z-Score des Log-Kurses steuert ihre Position gegenüber dem **Grundgewicht**:

| Abweichung vom Mittel | Gewicht |
|---|---|
| +Kappung (Vorgabe 2σ) | 0 % |
| 0 | Grundgewicht |
| −Kappung | doppeltes Grundgewicht |

Die Rohgewichte werden täglich auf 100 % renormiert (FX kennt keine Kasse). Der Backtest stellt das gegen
denselben Korb mit festen Grundgewichten: gleichgewichtet (`equal`) oder nach BIS-Handelsanteilen (`trade`).

**Vorlauf:** Das Signal braucht `lookback + smoothing + lag` Handelstage Historie – `python tasks/update_fx.py` lädt dafür immer die komplette Historie.

In [ ]:
import pandas as pd

from fx_analysis import quote, store

# --- Einstellungen -----------------------------------------------------------
PARAMS = dict(lookback=1008, cap=2.0, smoothing=15, lag=1)
BENCHMARK = "equal"   # "equal" = 1/N, "trade" = BIS-Handelsanteile
KOSTEN_BP = 5         # Handelskosten in Basispunkten auf den täglichen Umschlag
MIT_CARRY = True      # Leitzinsen (data/policy_rates.csv) gutschreiben, falls vorhanden
# -----------------------------------------------------------------------------

rates = store.load_rates() if MIT_CARRY else None
bt = quote.run(matrix, rates=rates, benchmark=BENCHMARK, cost_bp=KOSTEN_BP, **PARAMS)
print(f"Backtest {bt.index[0]:%d.%m.%Y} – {bt.index[-1]:%d.%m.%Y}, {len(bt)} Handelstage")
idx = pd.IndexSlice
quote.comparison(bt).style.format("{:+.2%}").format("{:.2f}", subset=idx[["Zeitraum (Jahre)", "Sharpe-Ratio", "Sortino-Ratio", "Calmar-Ratio"], :])

In [ ]:
from IPython.display import Image
from fx_analysis.config import QUOTE_BACKTEST_PNG, QUOTE_WEIGHTS_PNG

charts.render_quote_backtest(bt, QUOTE_BACKTEST_PNG, "Notebook-Lauf")
charts.render_quote_weights(bt, QUOTE_WEIGHTS_PNG, "Notebook-Lauf")
display(Image(QUOTE_BACKTEST_PNG), Image(QUOTE_WEIGHTS_PNG))

Aktueller Stand je Währung – sortiert von „teuer“ (Untergewicht) nach „billig“ (Übergewicht):

In [ ]:
quote.signal_status(bt).style.format({
    "Kurs": "{:.4f}", "z-Score": "{:+.2f}", "Gewichtsfaktor": "{:.2f}",
    "Grundgewicht": "{:.1%}", "Gewicht": "{:.1%}",
})